# Benchmark notebook

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%pip install loguru matplotlib numpy opencv-python pandas pandas_path pillow pyproj pytorch_lightning rasterio scikit-learn segmentation-models-pytorch torch torchvision tqdm transformers typer xarray xarray-spatial --quiet #--index-url https://download.pytorch.org/whl/cu124

In [ ]:
%env HF_ENDPOINT=https://hf-mirror.com

In [ ]:
import rasterio

from matplotlib import pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import torch

import warnings
warnings.filterwarnings("ignore")

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_capability())

x = torch.tensor([1.0, 2.0]).cuda()
print(x + x)

## Explore the data

First, we'll download all the data to a local folder. For instructions, see the competition's data download [page](). You'll need to join the competition in order to view the data page.

Data directory contents:
```
.
├── train_features
│   ├── train_chip_id_1
│   │   ├── B02.tif
│   │   ├── B03.tif
│   │   ├── B04.tif
│   │   └── B08.tif
│   └── ... 
├── train_metadata.csv
└── train_labels
    ├── train_chip_id_1.tif
    └── ...
```

In [ ]:
metadata_dir = Path.cwd().parent.parent.parent.resolve() / Path("data/GF1_B02_B03_B04_B08_Mask")
metadata_path = metadata_dir / "metadata.csv"
df = pd.read_csv(metadata_path)

def update_path(row):
    file_suffix = ".tif"
    filename = row["filename"] + file_suffix
    path = os.path.join(metadata_dir, filename)
    return path

df["path"] = df.apply(update_path, axis=1)
df.head(10)

The training data consists of 11,748 "chips". Each chip is imagery of a specific area captured at a specific point in time. There are four images associated with each chip in the competition data. Each image within a chip captures light from a different range of wavelengths, or "band". For example, the B02 band for each chip shows the strengh of visible blue light, which has a wavelength around 492 nanometers (nm). The bands provided are:

| Band | Description | Center wavelength |
|------|-------------|-------------------|
| B02 | Blue visible light | 497 nm |
| B03 | Green visible light | 560 nm |
| B04 | Red visible light | 665 nm |
| B08 | Near infrared light | 835 nm |

In [ ]:
BANDS = ["B02", "B03", "B04", "B08"]

In [ ]:
TRAIN_FEATURES = Path.cwd().parent.parent.parent.resolve() / Path("data/metadata")
TEST_FEATURES = TRAIN_FEATURES
assert TRAIN_FEATURES.exists()

### Metadata

Let's start by looking at the metadata for the train set, to understand what the images in this competition capture.

In [ ]:
csv_files = list(TRAIN_FEATURES.glob("*metadata.csv"))
csv_files

In [ ]:
training_targets_txt = TRAIN_FEATURES / "training_targets.txt"
with open(training_targets_txt, 'r', encoding='utf-8') as f:
    training_targets = [line.strip() for line in f if line.strip()]
training_targets

In [ ]:
selected_csv_files = [csv_file for csv_file in csv_files if csv_file.name.replace("_metadata.csv", "") in training_targets]
selected_csv_files

In [ ]:
df_list = []

for x in selected_csv_files:
    df = pd.read_csv(x)
    df_list.append(df)

train_meta = pd.concat(df_list, ignore_index=True).drop_duplicates()
len(train_meta)

In [ ]:
# how many different chip ids, locations, and datetimes are there?
train_meta[["chip_id", "location", "datetime"]].nunique()

We have one row per chip, and each `chip_id` is unique. There are columns for:
- `location`: General location of the chip
- `datetime`: Date and time the satellite image was captured
- `cloudpath`: All of the satellite images for this competition are hosted on Azure Blob Storage containers.

Let's take a look at the distribution of chips by location.

In [ ]:
train_location_counts = (
    train_meta.groupby("location")["chip_id"].nunique().sort_values(ascending=False)
)

In [ ]:
plt.figure(figsize=(12, 4))
train_location_counts.head(25).plot(kind="bar", color="lightgray")
plt.xticks(rotation=90)
plt.xlabel("Location")
plt.ylabel("Number of Chips")
plt.title("Number of Train Chips by Location (Top 25)")
plt.show()

The train and test images are from all over the world! Location names can be countries, cities, or broader regions. Sentinel-2 flies over the part of the Earth between 56° South and 82.8° North. The chips are mostly in Africa and South America, with some in Australia too.

We also have a timestamp for each chip. What is the time range in the data?

In [ ]:
train_meta["datetime"] = pd.to_datetime(train_meta["datetime"])
train_meta["year"] = train_meta.datetime.dt.year
train_meta.groupby("year")[["chip_id"]].nunique().sort_index().rename(
    columns={"chip_id": "chip_count"}
)

In [ ]:
train_meta["datetime"].min(), train_meta["datetime"].max()

In [ ]:
chips_per_locationtime = (
    train_meta.groupby(["location", "datetime"])[["chip_id"]]
    .nunique()
    .sort_values(by="chip_id", ascending=False)
    .rename(columns={"chip_id": "chip_count"})
)
chips_per_locationtime.head(10)

All of the images in the data were captured between February of 2018 and September of 2020. We can also see from the training data that many chips share the same location and time.

We know from the problem description that all of the locations in the test set are new, and there is no overlap between locations in the train set and the test set.

#### Images

Next, let's explore the actual satellite images - the star of the show for this challenge!

For convenience, let's first add the paths to all of the feature images per chip.

def add_paths(df, feature_dir, label_dir=None, bands=BANDS):
    """
    Given dataframe with a column for chip_id, returns a dataframe with a column
    added indicating the path to each band's TIF image as "{band}_path", eg "B02_path".
    A column is also added to the dataframe with paths to the label TIF, if the
    path to the labels directory is provided.
    """
    for band in bands:
        df[f"{band}_path"] = feature_dir / df["chip_id"] / f"{band}.tif"
        assert df[f"{band}_path"].path.exists().all()
    if label_dir is not None:
        df["label_path"] = label_dir / (df["chip_id"] + ".tif")
        assert df["label_path"].path.exists().all()

    return df

train_meta = add_paths(train_meta, TRAIN_FEATURES, TRAIN_LABELS)
train_meta.head()

Each image is a GeoTIFF, a raster image file that contains geographic metadata. This metadata can include coordinates, an affine transform, and a coordinate reference system (CRS) projection. The package rasterio makes it easy to interact with our geospatial raster data.

Lets look at the red visible band (B04) image for a random chip.

In [ ]:
example_chip = train_meta.head()
display(example_chip)

In [ ]:
example_chip = example_chip.iloc[0]
with rasterio.open(example_chip["B04_path"]) as img:
    chip_metadata = img.meta
    img_array = img.read(1)

chip_metadata

We can see that the features are single-band images, with a shape of 512 x 512. The pixel values for each image measure the strength of light reflected back to the satellite for the specific set of wavelengths in that band.

In [ ]:
# what does the image array look like?
print("Image array shape:", img_array.shape)
img_array

In [ ]:
np.isnan(img_array).sum()

In [ ]:
plt.imshow(img_array)
plt.title(f"B04 band for chip id {example_chip.chip_id}")
plt.show()

##### Coordinates

Using the metadata returned by `rasterio`, we can also get longitude and latitude coordinates.

In [ ]:
# longitude/latitude of image's center
with rasterio.open(example_chip["B04_path"]) as img:
    lon, lat = img.lnglat()
    bounds = img.bounds
print(f"Longitude: {lon}, latitude: {lat}")

bounds

We have the longitude and latitude of the center of the image, but the bounding box values look very different. That's because the bounding box is given in whatever coordinate reference system the image is projected in. We can convert the bounding box to longitude and latitude using `pyproj`.

In [ ]:
import pyproj

def lat_long_bounds(filepath):
    """Given the path to a GeoTIFF, returns the image bounds in latitude and
    longitude coordinates.

    Returns points as a tuple of (left, bottom, right, top)
    """
    with rasterio.open(filepath) as im:
        bounds = im.bounds
        meta = im.meta
    # create a converter starting with the current projection
    current_crs = pyproj.CRS(meta["crs"])
    crs_transform = pyproj.Transformer.from_crs(current_crs, current_crs.geodetic_crs)

    # returns left, bottom, right, top
    return crs_transform.transform_bounds(*bounds)

left, bottom, right, top = lat_long_bounds(example_chip["B04_path"])
print(f"Image coordinates (lat, long):\nStart: ({left}, {bottom})\nEnd: ({right}, {top})")

##### True color image

We can make a composite image from the three visible bands (blue, green, and red) to visualize a high-quality, true color image.

In [ ]:
import xarray
import xrspatial.multispectral as ms

def get_xarray(filepath):
    """Put images in xarray.DataArray format"""
    im_arr = np.array(Image.open(filepath))
    return xarray.DataArray(im_arr, dims=["y", "x"])

def true_color_img(chip):
    """Given the path to the directory of Sentinel-2 chip feature images,
    plots the true color image"""
    red = get_xarray(chip.B04_path)
    green = get_xarray(chip.B03_path)
    blue = get_xarray(chip.B02_path)

    return ms.true_color(r=red, g=green, b=blue)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
im = true_color_img(example_chip)
ax[0].imshow(im)
ax[0].set_title(f"True color image for chip id {example_chip.chip_id}")
label_im = Image.open(example_chip["label_path"])
ax[1].imshow(label_im)
ax[1].set_title(f"Chip {example_chip.chip_id} label")
plt.tight_layout()
plt.show()

In [ ]:
def display_random_chip(random_state):
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    random_chip = train_meta.sample(random_state=random_state).iloc[0]
    ax[0].imshow(true_color_img(random_chip))
    ax[0].set_title(f"Chip {random_chip.chip_id}\n(Location: {random_chip.location})")
    label_im = Image.open(random_chip.label_path)
    ax[1].imshow(label_im)
    ax[1].set_title(f"Chip {random_chip.chip_id} label")

    plt.tight_layout()
    plt.show()

display_random_chip(1)
display_random_chip(9)
display_random_chip(40)

From the example chips, we can see that there is a very big variation in the amount of cloud cover per chip.

### Plan the data
Summarize mean chip-level class ratios from metadata and build a chip-count planning table by label-ratio bins and optional groups.

In [ ]:
from benchmark.core.metadata_io import summarize_label_distribution

class_summary = summarize_label_distribution(train_meta)

class_summary

In [ ]:
from benchmark.core.metadata_io import plan_label_distribution

training_plan = plan_label_distribution(train_meta)

training_plan

In [ ]:
from benchmark.core.metadata_io import sample_train_metadata_by_label_bins

fast_dev_run = True
ratio = 0.01 if fast_dev_run else 1.0

train_meta = sample_train_metadata_by_label_bins(
    train_meta,
    keep_ratio={
        "(0.0, 0.01]":  0.50 * ratio,
        "(0.01, 0.05]": 1.00 * ratio,
        "(0.05, 0.2]":  1.00 * ratio,
        "(0.2, 0.5]":   1.00 * ratio,
        "(0.5, 0.8]":   0.75 * ratio,
        "(0.8, 0.95]":  0.50 * ratio,
        "(0.95, 1.0]":  0.05 * ratio,
        "default":      0.10 * ratio,
    },
    min_per_group=10 if not fast_dev_run else 1,
)

training_plan = plan_label_distribution(train_meta)

training_plan

### Split the data

To train our model, we want to separate the data into a "training" set and a "validation" set. That way we'll have a portion of labelled data that was not used in model training, which can give us a more accurate sense of how our model will perform on the competition test data.

We have chosen the simplest route, and split our training chips randomly into 15% test, 15% validation and 70% training.

In [ ]:
import random

random.seed(9)  # set a seed for reproducibility

# put 25%, 5% and 70% of chips into the test, validation and training set respectively
chip_ids = train_meta.chip_id.unique().tolist()
n = len(chip_ids)

random.shuffle(chip_ids)
train_ids = chip_ids[:round(n * 0.70)]
val_ids = chip_ids[round(n * 0.70):round(n * 0.95)]
test_ids = chip_ids[round(n * 0.95):]

train = train_meta[train_meta.chip_id.isin(train_ids)].copy().reset_index(drop=True)
val = train_meta[train_meta.chip_id.isin(val_ids)].copy().reset_index(drop=True)
test = train_meta[train_meta.chip_id.isin(test_ids)].copy().reset_index(drop=True)

test.shape, val.shape, train.shape

In [ ]:
# separate features from labels
feature_cols = ["chip_id"] + [f"{band}_path" for band in BANDS]

test_x = test[feature_cols].copy()
test_y = test[["chip_id", "label_path"]].copy()

val_x = val[feature_cols].copy()
val_y = val[["chip_id", "label_path"]].copy()

train_x = train[feature_cols].copy()
train_y = train[["chip_id", "label_path"]].copy()

In [ ]:
val_x.head()

In [ ]:
val_y.head()

### Build the model

In [ ]:
from pathlib import Path

model_name = "unet"
benchmark_dir = Path(model_name)

### Fit the model

In [ ]:
import os
import pytorch_lightning as pl
import warnings
warnings.filterwarnings("ignore")

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

torch.set_float32_matmul_precision("high")

In [ ]:
from benchmark.dataset import CloudDataModule
from benchmark.configs import Configs
from benchmark.models.cloud_model import CloudModel

config = Configs.balanced()

cloud_datamodule = CloudDataModule(
    config=config,
    x_train=train_x,
    y_train=train_y,
    x_val=val_x,
    y_val=val_y,
    x_test=test_x,
    y_test=test_y,
)

cloud_model = CloudModel(
    config=config,
)

callbacks = [
    pl.callbacks.ModelCheckpoint(
        dirpath="checkpoints",
        filename="epoch{epoch:02d}-val_avg_iou{val/avg_iou:.4f}",
        auto_insert_metric_name=False,
        monitor="val/avg_iou",
        mode="max",
        save_top_k=1,
        save_last=False,
        verbose=True,
    ),
    pl.callbacks.early_stopping.EarlyStopping(
        monitor="val/avg_iou",
        mode="max",
        patience=15,
        verbose=True,
    ),
    pl.callbacks.LearningRateMonitor(logging_interval="step"),
]

trainer = pl.Trainer(
    accelerator="auto",
    devices="auto",
    # fast_dev_run=1,
    callbacks=callbacks,
    max_epochs=100,
    precision="bf16",
    enable_checkpointing=True,
    enable_progress_bar=True,
    enable_model_summary=True,
)

In [ ]:
# Fit the model
last_ckpt = Path("checkpoints/epoch13-val_avg_iou0.7788.ckpt")
trainer.fit(
    model=cloud_model,
    datamodule=cloud_datamodule,
    # ckpt_path=last_ckpt if last_ckpt.exists() else None,
    # weights_only=False,
)

### Save the model

In [ ]:
# save the model
benchmark_assets_dir = benchmark_dir / "assets"
benchmark_assets_dir.mkdir(parents=True, exist_ok=True)

model_weight_path = benchmark_assets_dir / "cloud_model.pt"
torch.save(cloud_model.state_dict(), model_weight_path)

### Test the model

In [ ]:
# test the model
trainer.test(
    cloud_model,
    datamodule=cloud_datamodule,
    ckpt_path="best" if trainer.checkpoint_callback.best_model_path else None,
    weights_only=False,
)